# Task 3 — Research Results, Data Analysis and Scientific Visualization

**Project:** The Impact of Generative AI Assistance on Programming Learning Effectiveness Among Undergraduate Students  
**Authors (Task 2):** Yerkindik Ailen and Aitkali Dariga  
**Data:** **SYNTHETIC DEMONSTRATION ONLY** — not real participants, not evidence for hypotheses.

The three study conditions are No AI (baseline), AI Hints, and AI Solutions. The primary metric is independent programming task score (%) collected without AI assistance after practice. The attached demo log is fabricated to test processing; no real independent assessment has been conducted.

## 1. Configuration and data ingestion
Find the repository root and import the reusable processing modules. Fix the random seed for reproducibility.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'process_data.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'process_data.py').exists(), 'Run from repo root or notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.process_data import load_jsonl, validate_and_filter, aggregate, bootstrap_ci
from src.visualize import generate_figures
np.random.seed(42)
raw = load_jsonl(ROOT / 'data/raw/test_run.jsonl')
print('Raw records:', len(raw))
print('Data type:', sorted(raw.data_type.unique()))
display(raw.head(4))

Raw records: 60
Data type: ['synthetic']


,run_id,group,phase,task_load,practice_score_pct,independent_score_pct,completion_time_min,concept_score_pct,programming_errors,execution_success,data_type
0,no_ai-warmup-01,no_ai,warmup,1,73.4,69.0,50.72,76.5,1,True,synthetic
1,no_ai-warmup-02,no_ai,warmup,2,66.2,54.7,60.24,72.7,1,True,synthetic
2,no_ai-evaluation-01,no_ai,evaluation,1,76.4,72.4,26.43,69.2,3,True,synthetic
3,no_ai-evaluation-02,no_ai,evaluation,2,71.0,73.9,36.10,69.5,2,True,synthetic


## 2. Cold-start filtering and data validation
Exclude explicitly marked `warmup` runs from performance estimates. Preserve all evaluation runs for the execution-error and validity guardrails. Failed executions are excluded from successful-run score averages, but included in error rates.

In [2]:
evaluation, successful = validate_and_filter(raw)
print('Excluded warm-up runs:', (raw.phase == 'warmup').sum())
print('Evaluation runs:', len(evaluation))
print('Valid and successful analyzed runs:', len(successful))
print('Execution failures:', (~evaluation.execution_success.astype(bool)).sum())
display(evaluation.groupby('group', sort=False).agg(n=('run_id','size'), valid_rate=('valid','mean'), success_rate=('execution_success','mean')))

Excluded warm-up runs: 6
Evaluation runs: 54
Valid and successful analyzed runs: 53
Execution failures: 1


,n,valid_rate,success_rate
group,,,
no_ai,18,1.0,1.000000
ai_hints,18,1.0,1.000000
ai_solutions,18,1.0,0.944444


## 3. Statistical aggregation
Report mean, sample standard deviation, median, p25/p75, p95 and p99 for every metric. The 95% confidence intervals are obtained via a fixed-seed percentile bootstrap (3000 resamples per group and metric). Bootstrap CIs from small artificial datasets are illustrative only.

In [3]:
summary = aggregate(evaluation, successful)
summary_path = ROOT / 'data/processed/metrics_summary.csv'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_path, index=False, float_format='%.5f')
selected = ['group','n_analyzed','practice_score_pct_mean','independent_score_pct_mean',
            'independent_score_pct_sd','independent_score_pct_median','independent_score_pct_p95',
            'independent_score_pct_p99','independent_score_pct_ci_low','independent_score_pct_ci_high',
            'completion_time_min_mean','execution_error_rate_pct','valid_results_rate_pct']
display(summary[selected].round(2))
print('Saved:',summary_path.relative_to(ROOT))

,group,n_analyzed,practice_score_pct_mean,independent_score_pct_mean,independent_score_pct_sd,independent_score_pct_median,independent_score_pct_p95,independent_score_pct_p99,independent_score_pct_ci_low,independent_score_pct_ci_high,completion_time_min_mean,execution_error_rate_pct,valid_results_rate_pct
0,no_ai,18,70.59,64.75,6.59,65.25,73.56,73.83,61.71,67.74,36.37,0.00,100.0
1,ai_hints,18,87.88,79.16,6.66,80.45,86.41,87.36,76.02,81.87,25.57,0.00,100.0
2,ai_solutions,17,77.26,62.13,5.40,62.90,70.70,72.94,59.46,64.60,20.57,5.56,100.0


Saved: data/processed/metrics_summary.csv


## 4. Baseline comparison and limitations
Differences are descriptive: group means are compared with the No AI baseline. This notebook makes **no** statistical significance or causal claim. Synthetic runs are not independent student observations.

In [4]:
baseline = summary.set_index('group').loc['no_ai']
comparison = summary[['group','independent_score_pct_mean','completion_time_min_mean']].copy()
comparison['independent_score_difference_vs_no_ai_pp'] = comparison['independent_score_pct_mean'] - baseline['independent_score_pct_mean']
comparison['completion_time_difference_vs_no_ai_min'] = comparison['completion_time_min_mean'] - baseline['completion_time_min_mean']
display(comparison.round(2))

,group,independent_score_pct_mean,completion_time_min_mean,independent_score_difference_vs_no_ai_pp,completion_time_difference_vs_no_ai_min
0,no_ai,64.75,36.37,0.00,0.00
1,ai_hints,79.16,25.57,14.41,-10.81
2,ai_solutions,62.13,20.57,-2.62,-15.80


## 5. Scientific visualization and figure export
Figure 1: mean completion time (minutes) by programming task difficulty; bootstrap 95% CIs. Figure 2: independent task score (%) boxplots, with mean and bootstrap 95% CI overlay. Accessible colors, different markers and line styles, explicit axes and physical units are used. Exports: PDF and PNG at 320 DPI.

In [5]:
files = generate_figures(successful, ROOT / 'reports/figures')
print('Saved:', *[str(f.relative_to(ROOT)) for f in files],sep='\n- ')

Saved:
- reports/figures/fig1_latency_vs_load.pdf
- reports/figures/fig2_distribution_boxplot.pdf


## 6. Integrity checks and interpretation
The output below verifies that warm-up runs were removed, the JSONL has unique run IDs, the aggregate contains three groups, and that each figure was exported in both requested formats. Any observed difference reflects fabricated data, not evidence about student learning.

In [6]:
assert set(summary.group) == {'no_ai','ai_hints','ai_solutions'}
assert successful.phase.eq('evaluation').all()
assert raw.run_id.is_unique
assert summary.n_analyzed.sum() == len(successful)
for stem in ('fig1_latency_vs_load','fig2_distribution_boxplot'):
    for extension in ('.pdf','.png'):
        assert (ROOT/'reports/figures'/f'{stem}{extension}').is_file()
print('All integrity checks passed. SYNTHETIC demonstration only.')

All integrity checks passed. SYNTHETIC demonstration only.
